<a href="https://colab.research.google.com/github/coolboy-dev/Amazon-ML-challenge-2025/blob/main/Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Importing the dataset
import pandas as pd
df = pd.read_csv('/content/drive/My Drive/amazon/images_test/processed_train.csv')
print(f"DataFrame Headers: {list(df.columns)}")

In [ ]:
# The -q flag means 'quiet' so it doesn't print all 75,000 filenames
!unzip -q "/content/drive/My Drive/amazon/images.zip" -d "/content/images/"

In [ ]:
# Install dependencies
!pip install torch torchvision pillow pandas numpy tqdm scikit-learn

# Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")


class DINOv2Extractor:
    def __init__(self, model_size='small', batch_size=32):
        self.device = device
        self.batch_size = batch_size
        model_name = f'dinov2_vit{model_size[0]}14'
        self.model = torch.hub.load('facebookresearch/dinov2', model_name, trust_repo=True)
        self.model = self.model.to(self.device).eval()
        self.use_amp = torch.cuda.is_available()
        embed_dims = {'small': 384, 'base': 768, 'large': 1024, 'giant': 1536}
        self.embedding_dim = embed_dims[model_size]
        self.transform = transforms.Compose([
            transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def load_image(self, img_path):
        try:
            return self.transform(Image.open(img_path).convert('RGB'))
        except Exception:
            return torch.zeros(3, 224, 224)

    def extract_batch(self, batch_images):
        batch_tensor = torch.stack(batch_images).to(self.device)
        with torch.no_grad():
            if self.use_amp:
                with torch.cuda.amp.autocast():
                    return self.model(batch_tensor).cpu().numpy()
            return self.model(batch_tensor).cpu().numpy()

    def process_dataset(self, df, image_dir, save_path='dinov2_embeddings.pkl'):
        image_dir = Path(image_dir)
        embeddings_list, prices_list, product_ids_list = [], [], []
        failed_count = 0
        batch_images, batch_data = [], []

        pbar = tqdm(total=len(df), desc="Extracting DINOv2 embeddings")

        for idx, row in df.iterrows():
            img_path = image_dir / row['image_filename']
            if img_path.exists():
                batch_images.append(self.load_image(str(img_path)))
                batch_data.append({'price': row['price'], 'id': row['id']})
            else:
                failed_count += 1
                pbar.update(1)
                continue

            if len(batch_images) == self.batch_size:
                batch_embeddings = self.extract_batch(batch_images)
                embeddings_list.append(batch_embeddings)
                for d in batch_data:
                    prices_list.append(d['price'])
                    product_ids_list.append(d['id'])
                batch_images, batch_data = [], []
                pbar.update(self.batch_size)
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        if batch_images:
            batch_embeddings = self.extract_batch(batch_images)
            embeddings_list.append(batch_embeddings)
            for d in batch_data:
                prices_list.append(d['price'])
                product_ids_list.append(d['id'])
            pbar.update(len(batch_images))

        pbar.close()

        embeddings = np.vstack(embeddings_list)
        data = {
            'embeddings': embeddings,
            'prices': np.array(prices_list),
            'product_ids': np.array(product_ids_list),
            'embedding_dim': self.embedding_dim,
            'failed_count': failed_count,
        }

        with open(save_path, 'wb') as f:
            pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

        print(f"Saved {len(embeddings)} embeddings to {save_path} ({failed_count} failed)")
        return data


# Configuration
DRIVE = '/content/drive/My Drive/amazon/images_test/'
TRAIN_CSV = DRIVE + 'processed_train.csv'
IMAGE_DIR = '/content/images/images/'
OUTPUT_FILE = DRIVE + 'dinov2_embeddings.pkl'

df = pd.read_csv(TRAIN_CSV)
print(f"Loaded {len(df)} products")

extractor = DINOv2Extractor(model_size='small', batch_size=32)
data = extractor.process_dataset(df, IMAGE_DIR, OUTPUT_FILE)


In [ ]:
import torch
import pandas as pd
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm


DRIVE = '/content/drive/My Drive/amazon/images_test/'
TRAIN_CSV = '/content/train.csv'
OUTPUT_FILE = (DRIVE + 'text_embeddings_minilm.pkl')
MODEL_NAME = 'all-MiniLM-L6-v2'


def parse_and_clean_text(content):
    """
    Parses the multi-line catalog_content string into a single, clean text string.
    """
    if not isinstance(content, str):
        return ""

    lines = [line.strip() for line in content.split('\n') if line.strip()]

    text_parts = []


    for line in lines:
        if line.startswith("Item Name:"):
            text_parts.append(line.replace("Item Name:", "").strip())
        elif line.startswith("Bullet Point"):

            parts = line.split(":", 1)
            if len(parts) > 1:
                text_parts.append(parts[1].strip())
        elif line.startswith("Product Description:"):
            text_parts.append(line.replace("Product Description:", "").strip())


    return ". ".join(part for part in text_parts if part)



if __name__ == "__main__":
    print(f"📂 Loading data from '{TRAIN_CSV}'...")
    df = pd.read_csv(TRAIN_CSV)

    print("📝 Parsing and cleaning text data...")
    # Use tqdm to show progress for the apply function
    tqdm.pandas(desc="Cleaning text")
    df['cleaned_text'] = df['catalog_content'].progress_apply(parse_and_clean_text)

    print("\n✅ Text cleaning complete. Here's a sample:")
    print("-" * 50)
    print(df['cleaned_text'].iloc[1])
    print("-" * 50)

    print(f"\n🧠 Loading SentenceTransformer model: '{MODEL_NAME}'")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = SentenceTransformer(MODEL_NAME, device=device)
    print(f"✓ Model loaded on {device}")

    print("\n🚀 Generating text embeddings...")
    # The .encode() method is highly optimized for this task
    text_embeddings = model.encode(
        df['cleaned_text'].tolist(),
        show_progress_bar=True,
        batch_size=128  # You can adjust this based on your GPU memory
    )

    print(f"\n📊 Text embeddings created with shape: {text_embeddings.shape}")

    # --- 4. SAVE THE RESULTS ---
    # We save embeddings and their corresponding IDs to map them later
    data_to_save = {
        'sample_ids': df['sample_id'].tolist(),
        'embeddings': text_embeddings
    }

    print(f"\n💾 Saving embeddings to '{OUTPUT_FILE}'...")
    with open(OUTPUT_FILE, 'wb') as f:
        pickle.dump(data_to_save, f)

    print("\n🎉 Success! Text embedding process is complete.")

In [ ]:
import pandas as pd
import numpy as np
import pickle

# --- 1. CONFIGURATION --
DRIVE = '/content/drive/My Drive/amazon/images_test/'
IMAGE_EMBEDDINGS_FILE = (DRIVE + 'dinov2_embeddings.pkl')
TEXT_EMBEDDINGS_FILE = (DRIVE + 'text_embeddings_minilm.pkl')
FINAL_OUTPUT_FILE = (DRIVE + 'final_multimodal_dataset.pkl')

# --- 2. LOAD BOTH DATASETS ---
print("📂 Loading image embeddings...")
with open(IMAGE_EMBEDDINGS_FILE, 'rb') as f:
    # Note: Your DINOv2 script saved 'product_ids', which are the sample_ids
    image_data = pickle.load(f)
df_image = pd.DataFrame({
    'sample_id': image_data['product_ids'],
    'price': image_data['prices'],
    'image_embedding': list(image_data['embeddings'])
})
print(f"✓ Found {len(df_image)} image embeddings.")

print("\n📂 Loading text embeddings...")
with open(TEXT_EMBEDDINGS_FILE, 'rb') as f:
    text_data = pickle.load(f)
df_text = pd.DataFrame({
    'sample_id': text_data['sample_ids'],
    'text_embedding': list(text_data['embeddings'])
})
print(f"✓ Found {len(df_text)} text embeddings.")


# --- 3. MERGE THE DATAFRAMES ---
print("\n🔗 Merging datasets using 'sample_id'...")
# An 'inner' merge ensures we only keep IDs that exist in BOTH files.
final_df = pd.merge(df_image, df_text, on='sample_id', how='inner')


print(f"✓ Merge complete. Final dataset has {len(final_df)} matched products.")


# --- 4. COMBINE THE EMBEDDING VECTORS ---
print("\n🧬 Concatenating image and text vectors...")
# We'll loop through the merged dataframe and stack the embeddings horizontally.
# np.hstack is perfect for this.
combined_embeddings = np.hstack((
    np.vstack(final_df['image_embedding']),
    np.vstack(final_df['text_embedding'])
))

print(f"✓ Concatenation complete.")
print(f"  Final embedding dimension: {combined_embeddings.shape[1]}")


# --- 5. SAVE THE FINAL DATASET ---
# This dictionary contains everything needed for Day 2.
final_data_to_save = {
    'sample_ids': final_df['sample_id'].tolist(),
    'embeddings': combined_embeddings,
    'prices': final_df['price'].values
}

print(f"\n💾 Saving final dataset to '{FINAL_OUTPUT_FILE}'...")
with open(FINAL_OUTPUT_FILE, 'wb') as f:
    pickle.dump(final_data_to_save, f)

print("\n🎉 All data processing is complete! You are officially ready to train your model.")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pickle
from sklearn.model_selection import train_test_split

# --- 1. CONFIGURATION & HYPERPARAMETERS ---

DRIVE = '/content/drive/My Drive/amazon/images_test/'
DATASET_FILE = (DRIVE + 'final_multimodal_dataset.pkl')
LEARNING_RATE = 0.001
BATCH_SIZE = 256
EPOCHS = 20 # Train for a few epochs; this task converges quickly
TEST_SPLIT_SIZE = 0.2 # Use 20% of data for validation

# --- 2. LOAD AND PREPARE DATA ---
print(f"📂 Loading final dataset from '{DATASET_FILE}'...")
with open(DATASET_FILE, 'rb') as f:
    data = pickle.load(f)

embeddings = data['embeddings'].astype(np.float32)
prices = data['prices'].astype(np.float32).reshape(-1, 1) # Reshape for PyTorch

print(f"✓ Loaded {len(embeddings)} total samples.")
print(f"   Embedding dimension: {embeddings.shape[1]}")

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    embeddings, prices, test_size=TEST_SPLIT_SIZE, random_state=42
)
print(f"\nSplit data into {len(X_train)} training and {len(X_val)} validation samples.")

# Create PyTorch Datasets and DataLoaders
train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
val_dataset = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# --- 3. DEFINE THE REGRESSION HEAD MODEL ---
# Input dimension is 384 (DINOv2) + 384 (MiniLM) = 768
INPUT_DIM = embeddings.shape[1]

class RegressionHead(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 256)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3) # Dropout helps prevent overfitting
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 1) # Final output is a single number (the price)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# --- 4. TRAINING SETUP ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = RegressionHead(INPUT_DIM).to(device)
# Use L1Loss because it directly corresponds to Mean Absolute Error (MAE)
loss_function = nn.L1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

print(f"\n🚀 Starting training on {device}...")

# --- 5. TRAINING LOOP ---
for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0
    for batch_embeddings, batch_prices in train_loader:
        batch_embeddings, batch_prices = batch_embeddings.to(device), batch_prices.to(device)

        # Forward pass
        outputs = model(batch_embeddings)
        loss = loss_function(outputs, batch_prices)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # --- VALIDATION ---
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch_embeddings, batch_prices in val_loader:
            batch_embeddings, batch_prices = batch_embeddings.to(device), batch_prices.to(device)
            outputs = model(batch_embeddings)
            loss = loss_function(outputs, batch_prices)
            total_val_loss += loss.item()

    avg_val_mae = total_val_loss / len(val_loader)

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Validation MAE: ${avg_val_mae:.4f}")

print("\n🎉 Training complete!")

In [ ]:
MODEL_PATH = DRIVE + 'regression_head_model.pth'
torch.save(model.state_dict(), MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")


In [ ]:
import pandas as pd
import requests
from PIL import Image
import io
from pathlib import Path
from tqdm.auto import tqdm

# --- CONFIGURATION ---
CSV_FILE = '/content/sample_test.csv'
IMAGE_DIR = '/content/test_images/' # The folder where we'll save the images

# --- MAIN SCRIPT ---
# 1. Create the directory if it doesn't exist
Path(IMAGE_DIR).mkdir(exist_ok=True)
print(f"📁 Images will be saved to '{IMAGE_DIR}'")

# 2. Load the CSV
df = pd.read_csv(CSV_FILE)
print(f"Found {len(df)} image links to download.")

# 3. Loop, download, and save
success_count = 0
fail_count = 0

for index, row in tqdm(df.iterrows(), total=len(df), desc="Downloading images"):
    image_url = row['image_link']
    # Use the sample_id as the filename for easy mapping
    image_filename = f"{row['sample_id']}.jpg"
    save_path = Path(IMAGE_DIR) / image_filename

    try:
        # Send a request to the URL
        response = requests.get(image_url, timeout=10)
        response.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)

        # Open the image from the response content
        image = Image.open(io.BytesIO(response.content)).convert("RGB")

        # Save the image as a JPEG
        image.save(save_path, 'jpeg')
        success_count += 1

    except Exception as e:
        fail_count += 1
        # Uncomment the line below if you want to see errors as they happen
        # print(f"Failed to download {image_url}: {e}")

print("\n--- DOWNLOAD COMPLETE ---")
print(f"✅ Success: {success_count}/{len(df)}")
print(f"❌ Failed:  {fail_count}/{len(df)}")

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import pickle
import warnings
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')

# Configuration
TEST_CSV = 'sample_test.csv'
IMAGE_DIR = 'test_images/'
MODEL_SAVE_PATH = DRIVE + 'regression_head_model.pth'
DINO_MODEL_SIZE = 'small'
TEXT_MODEL_NAME = 'all-MiniLM-L6-v2'
BATCH_SIZE = 64
INPUT_DIM = 768

device = 'cuda' if torch.cuda.is_available() else 'cpu'


class DINOv2Extractor:
    def __init__(self, model_size='small', batch_size=32):
        self.device = torch.device(device)
        self.batch_size = batch_size
        model_name = f'dinov2_vit{model_size[0]}14'
        self.model = torch.hub.load('facebookresearch/dinov2', model_name, trust_repo=True).to(self.device).eval()
        self.transform = transforms.Compose([
            transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def load_image(self, img_path):
        try:
            return self.transform(Image.open(img_path).convert('RGB'))
        except Exception:
            return torch.zeros(3, 224, 224)

    def extract_embeddings(self, df, image_dir):
        image_dir = Path(image_dir)
        all_embeddings = {}
        batch_images, batch_ids = [], []

        for _, row in tqdm(df.iterrows(), total=len(df), desc="Image embeddings"):
            img_path = image_dir / f"{row['sample_id']}.jpg"
            if img_path.exists():
                batch_images.append(self.load_image(str(img_path)))
                batch_ids.append(row['sample_id'])

            if len(batch_images) == self.batch_size:
                batch_tensor = torch.stack(batch_images).to(self.device)
                with torch.no_grad(), torch.cuda.amp.autocast():
                    embeddings = self.model(batch_tensor).cpu().numpy()
                for i, sid in enumerate(batch_ids):
                    all_embeddings[sid] = embeddings[i]
                batch_images, batch_ids = [], []

        if batch_images:
            batch_tensor = torch.stack(batch_images).to(self.device)
            with torch.no_grad(), torch.cuda.amp.autocast():
                embeddings = self.model(batch_tensor).cpu().numpy()
            for i, sid in enumerate(batch_ids):
                all_embeddings[sid] = embeddings[i]

        return all_embeddings


def parse_and_clean_text(content):
    if not isinstance(content, str):
        return ""
    lines = [line.strip() for line in content.split('\n') if line.strip()]
    text_parts = []
    for line in lines:
        if line.startswith("Item Name:"):
            text_parts.append(line.replace("Item Name:", "").strip())
        elif line.startswith("Bullet Point"):
            parts = line.split(":", 1)
            if len(parts) > 1:
                text_parts.append(parts[1].strip())
        elif line.startswith("Product Description:"):
            text_parts.append(line.replace("Product Description:", "").strip())
    return ". ".join(p for p in text_parts if p)


class RegressionHead(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.net(x)


# Load test data
df_test = pd.read_csv(TEST_CSV)

# Image embeddings
dino_extractor = DINOv2Extractor(model_size=DINO_MODEL_SIZE, batch_size=BATCH_SIZE)
image_embeddings_dict = dino_extractor.extract_embeddings(df_test, IMAGE_DIR)
df_image = pd.DataFrame(list(image_embeddings_dict.items()), columns=['sample_id', 'image_embedding'])

# Text embeddings
df_test['cleaned_text'] = df_test['catalog_content'].apply(parse_and_clean_text)
text_model = SentenceTransformer(TEXT_MODEL_NAME, device=device)
text_embeddings = text_model.encode(df_test['cleaned_text'].tolist(), show_progress_bar=True, batch_size=BATCH_SIZE)
df_text = pd.DataFrame({'sample_id': df_test['sample_id'], 'text_embedding': list(text_embeddings)})

# Merge and run inference
df_merged = pd.merge(df_image, df_text, on='sample_id', how='inner')
combined_embeddings = np.hstack((
    np.vstack(df_merged['image_embedding']),
    np.vstack(df_merged['text_embedding'])
)).astype(np.float32)

model = RegressionHead(INPUT_DIM).to(device)
model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.eval()

test_tensor = torch.from_numpy(combined_embeddings).to(device)
with torch.no_grad():
    dollar_predictions = torch.expm1(model(test_tensor)).cpu().numpy().flatten()

predictions = {sid: pred for sid, pred in zip(df_merged['sample_id'], dollar_predictions)}

df_test['price'] = df_test['sample_id'].map(predictions).fillna(0)
df_test.to_csv(TEST_CSV, index=False)
print(f"Saved predictions to {TEST_CSV}")
